# Hands-on Task: Build a 2-Step Agent

Runs several varied messages through one agent function, which decides to call a tool (possibly several) or answer directly, and prints its reasoning either way.

## Setup
Store your key in Colab **Secrets** (key icon, left sidebar) as `OPENAI_API_KEY`.

In [ ]:
!pip install -q openai

In [ ]:
from openai import OpenAI
from google.colab import userdata
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

In [ ]:
!pip install -q requests

In [ ]:
NTFY_TOPIC = "ojasa-mirai-agent-demo-CHANGE-ME"
import requests
def notify_admissions(message):
    r = requests.post(f"https://ntfy.sh/{NTFY_TOPIC}", data=message.encode("utf-8"))
    return f"notified admissions (status {r.status_code})"
def get_next_batch_date():
    return "November 15, 2026"
def log_feedback(feedback):
    return f"logged: {feedback}"

In [ ]:
tools = [
    {"type": "function", "function": {"name": "get_next_batch_date", "description": "Get the start date of the next AI Engineer course batch", "parameters": {"type": "object", "properties": {"reason": {"type": "string"}}, "required": ["reason"]}}},
    {"type": "function", "function": {"name": "log_feedback", "description": "Record a student's feedback about the course", "parameters": {"type": "object", "properties": {"feedback": {"type": "string"}, "reason": {"type": "string"}}, "required": ["feedback", "reason"]}}},
    {"type": "function", "function": {"name": "notify_admissions", "description": "Notify the admissions team to follow up with a prospective student", "parameters": {"type": "object", "properties": {"message": {"type": "string"}, "reason": {"type": "string"}}, "required": ["message", "reason"]}}},
]

In [ ]:
import json
def run_agent(message):
    response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": message}], tools=tools)
    msg = response.choices[0].message
    if not msg.tool_calls:
        print(f"[{message}]\n  -> answered directly: {msg.content}")
        return
    for call in msg.tool_calls:
        args = json.loads(call.function.arguments)
        print(f"[{message}]\n  -> called {call.function.name}, reason: {args['reason']}")

In [ ]:
full_message = ("I just finished the AI Engineer course and loved it, especially the RAG module "
                "-- though I'd have liked more time on evaluation. Also, a friend of mine wants to join "
                "the next batch, can someone from your team reach out to them? And do you know when "
                "the next batch starts?")
messages = [
    full_message,
    "What's the capital of France?",
    "When's the next batch starting?",
]
for m in messages:
    run_agent(m)

## How to Extend
Actually execute the tool calls (not just print the reason) using the real functions defined above, including a real notify_admissions call.

## Done When
At least 3 varied messages run, each either calling the right tool(s) with a sensible reason, or answering directly when no tool was needed.